<a href="https://colab.research.google.com/github/gibsonx/jlpt_simulator/blob/dev/graphs/n3/outliner.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
if 'google.colab' in str(get_ipython()):
    !git clone https://github.com/gibsonx/jlpt_simulator.git
    %cd jlpt_simulator
    !git checkout dev
    !apt-get install python3-dev graphviz libgraphviz-dev pkg-config
    !pip install -r requirements.txt
else:
  print('Not running on CoLab')

Not running on CoLab


In [2]:
import json
import logging
import random
import time
import pandas as pd
import yaml
import inspect
from tqdm import tqdm
import os
from libs.Logger import logger
from datetime import datetime
from docx import Document
from html4docx import HtmlToDocx
import uuid
from libs.CosmosMongoDB import CosmosMongoDB
from libs.LLMs import *
from IPython.display import display, Markdown, HTML
import datetime
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
from libs.Utils import render_to_html,collect_vocabulary,_load_vocab_and_resources
from graphs.common.Schema import Outline
from langchain_core.prompts import ChatPromptTemplate
from graphs.common.ExamGenerator import ExamGenerator
load_dotenv()

from graphs.common.TaskRunner import TaskRunner

# N1 Level Exam

In [3]:
# runner = TaskRunner(level="N1", exam_type="fast_exam")
# n1_outline, n1_exam_paper = runner.run()

## N1 Outline Preview

In [4]:
# display(Markdown(n1_outline.as_str))

## N1 HTML Result

In [5]:
# html_output = render_to_html(n1_exam_paper['sections'])
# display(HTML(html_output))
# timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
# filename = f"./output/jlpt_simulator/JLPT_{timestamp}.html"

# with open(filename, "w", encoding="utf-8") as file:
#     file.write(html_output)

# N2 Level Exam

In [6]:
runner = TaskRunner(level="N2", exam_type="full_exam")
n2_outline, n2_exam_paper = runner.run()

2025-11-10 01:32:16,349 - INFO - jlpt - Module 'graphs.n2.outliner' imported successfully.
2025-11-10 01:32:16,349 - INFO - Module 'graphs.n2.outliner' imported successfully.
2025-11-10 01:32:32,644 - INFO - HTTP Request: POST https://ai-rolandaws880125ai409947751408.openai.azure.com/openai/deployments/gpt-4.1/chat/completions?api-version=2025-01-01-preview "HTTP/1.1 200 OK"
2025-11-10 01:32:32,667 - INFO - jlpt - Outline of the exam:

# 日本語能力試験N2 模擬試験問題

## 第1部：語彙

### kanji_reading

5問（名詞2、動詞1、形容詞1、副詞1）。80%は難易度が非常に高い語彙を使用。

- **嚔**
- **嚴密**
- **潰す**
- **曖昧**
- **漸く**

### write_kanji

5問（名詞2、動詞1、形容詞1、副詞1）。80%は難易度が非常に高い語彙を使用。

- **ひゃっかじてん**
- **しゅうりょう**
- **くだける**
- **しつこい**
- **まもなく**

### words_collocation

3問（名詞1、動詞1、形容詞1）。80%は難易度が非常に高い語彙を使用。

- **ずいひつ**
- **ふるまう**
- **くだらない**

### word_meaning

7問（名詞2、動詞2、形容詞2、副詞1）。80%は難易度が非常に高い語彙を使用。

- **かんちょう**
- **せいさく**
- **つっこむ**
- **ふりむく**
- **そそっかしい**
- **きよい**
- **ひとりでに**

### synonym_substitution

5問（名詞2、動詞1、形容詞1、副詞1）。80%は難易度が非常に高い語彙を使用。

## N2 Outline Preview

In [7]:
display(Markdown(n2_outline.as_str))

# 日本語能力試験N2 模擬試験問題

## 第1部：語彙

### kanji_reading

5問（名詞2、動詞1、形容詞1、副詞1）。80%は難易度が非常に高い語彙を使用。

- **嚔**
- **嚴密**
- **潰す**
- **曖昧**
- **漸く**

### write_kanji

5問（名詞2、動詞1、形容詞1、副詞1）。80%は難易度が非常に高い語彙を使用。

- **ひゃっかじてん**
- **しゅうりょう**
- **くだける**
- **しつこい**
- **まもなく**

### words_collocation

3問（名詞1、動詞1、形容詞1）。80%は難易度が非常に高い語彙を使用。

- **ずいひつ**
- **ふるまう**
- **くだらない**

### word_meaning

7問（名詞2、動詞2、形容詞2、副詞1）。80%は難易度が非常に高い語彙を使用。

- **かんちょう**
- **せいさく**
- **つっこむ**
- **ふりむく**
- **そそっかしい**
- **きよい**
- **ひとりでに**

### synonym_substitution

5問（名詞2、動詞1、形容詞1、副詞1）。80%は難易度が非常に高い語彙を使用。

- **かんちがい**
- **じばん**
- **あきれる**
- **そうぞうしい**
- **すっきり**

### word_usage

5問（名詞3、動詞2）。80%は難易度が非常に高い語彙を使用。名詞は漢字表記。

- **領収書**
- **概論**
- **名作**
- **潰れる**
- **振り仮名**

## 第2部：文法

### sentence_grammar

12問（敬語1、副詞1、助詞1、その他9文型）。各設問は異なるトピックと文法を組み合わせる。

- **レストランで食べ物を注文する**お～願う
- **週末の予定について話す**いよいよ
- **交通手段について話す**に限る
- **健康とフィットネスについて話す**ざるを得ない
- **家族について話す**ものだから
- **趣味について話す**ばかりか
- **旅行の計画について話す**に際して
- **仕事のプロジェクトについて話す**に基づいて
- **最近の映画について話す**かのように
- **教育について話す**に応じて
- **環境問題について話す**にもかかわらず
- **将来の抱負について話す**限り

### sentence_sort

5問。各設問は異なるトピックと文法を組み合わせる。

- **家事の分担について話す**てこそ
- **支払い方法について話す**に加えて
- **趣味の道具やギアについて話す**だけあって
- **おすすめの旅行先について話す**にしたら
- **ボランティア活動について話す**ものの

### sentence_structure

1問。文章全体の内容を考えて、文中の空欄に最もよいものを選ぶ。

- **異文化交流について話す**に関わる、ばかりに、ものなら、しかも

## 第3部：読解

### short_passage_narrative_read

問題1-1：1記事。

- **ショッピング体験を説明する**

### short_passage_mail_read

問題1-2：1記事。

- **友人へのプレゼント選びについて話す**

### short_passage_narrative_read

問題1-3：1記事。

- **家の改善について話す**

### short_passage_notification_read

問題1-4：1記事。

- **公共施設の利用方法について話す**

### short_passage_narrative_read

問題1-5：1記事。

- **最近のニュースについて意見を述べる**

### midsize_passage_read

問題2：(1)(2)各2記事、計4記事。

- **健康診断や医者への訪問について話す**
- **地元の観光名所について話す**
- **キャリア目標について話す**
- **日本の祭りや文化イベントについて話す**

### comprehensive_reading

問題3：(1)(2)各1記事、計2記事。

- **家事の分担について話す**
- **地域社会への貢献について話す**

### long_passage_read

問題4：1記事。

- **技術について話す**

### info_retrieval

問題5：1記事。

- **海外旅行の体験を話す**

## 第4部：聴解

### topic_understanding_txt

問題1：5問。各設問は異なるトピック。

- **店で価格を尋ねる**
- **割引交渉**
- **道を尋ねる**
- **バスの時刻表を尋ねる**
- **電車の切符を買う**

### keypoint_understanding

問題2：4問。各設問は異なるトピック。

- **ペットについて話す**
- **ガーデニングについて話す**
- **ファッションとスタイルについて話す**
- **引っ越しの準備について話す**

### summary_understanding

問題3：5問。各設問は異なるトピック。

- **食事の好みについて話す**
- **料理を褒める**
- **お気に入りのレストランについて話す**
- **地元の食べ物や特産品について話す**
- **季節のイベントや活動について話す**

### immediate_ack

問題4：11問。各設問は異なるトピック。

- **友人との日常会話**
- **新しいスキルを学ぶ計画について話す**
- **学校や職場での一日の流れについて話す**
- **好きな季節について話す**
- **本について話す**
- **スポーツについて話す**
- **音楽について話す**
- **芸術と文化について話す**
- **個人的な成果について話す**
- **課題と解決策について話す**
- **将来の抱負について話す**

### comprehensive_expression_listen_answer

問題5-1：1問。

- **言語学習のコツについて話す**

### comprehensive_expression_show_answer

問題5-2：2問。

- **日本語学習の目的について話す**
- **子供の教育について話す**

## N2 Exam Result

In [8]:
html_output = render_to_html(n2_exam_paper['sections'])
display(HTML(html_output))

発表者,訪問国,主な体験内容,発表予定時間
佐藤美咲,イタリア,現地の家庭にホームステイし、伝統料理を学んだ。美術館や市場も訪問。,13:10～13:35
王俊,カナダ,語学学校で英語を勉強しながら、週末に自然公園を巡った。現地の友人と交流。,13:40～14:05
田中健太,タイ,ボランティア活動に参加し、子どもたちと一緒に学習支援をした。伝統的な祭りも体験。,14:10～14:35
マリア・ゴメス,スペイン,大学の短期プログラムで現地学生と共同研究。休日には歴史的な街を散策。,14:40～15:05
開始時間,場所,参加費,内容
15:30,駅前ビル2階「カフェ・ルミエール」,"1,500円",軽食・飲み物付き。発表者との交流や質問が可能。


In [9]:
timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
filename = f"./output/JLPT_N2_{timestamp}.html"

with open(filename, "w", encoding="utf-8") as file:
    file.write(html_output)

# N3 Level Exam

In [10]:
runner = TaskRunner(level="N3", exam_type="full_exam")
n3_outline, n3_exam_paper = runner.run()

2025-11-10 01:43:05,528 - INFO - jlpt - Module 'graphs.n3.outliner' imported successfully.
2025-11-10 01:43:05,528 - INFO - Module 'graphs.n3.outliner' imported successfully.
2025-11-10 01:43:20,465 - INFO - HTTP Request: POST https://ai-rolandaws880125ai409947751408.openai.azure.com/openai/deployments/gpt-4.1/chat/completions?api-version=2025-01-01-preview "HTTP/1.1 200 OK"
2025-11-10 01:43:20,480 - INFO - jlpt - Outline of the exam:

# 日本語能力試験 JLPT N3 模擬試験

## 第1部：語彙

### kanji_reading

8問：名詞3問、動詞3問、形容詞1問、副詞1問。80%は難易度が非常に高い語彙を使用。すべて漢字表記。

- **ぶんせき**
- **してん**
- **ろんぶん**
- **つかむ**
- **まじる**
- **あきらめる**
- **しんせん**
- **たまたま**

### write_kanji

6問：名詞2問、動詞2問、形容詞1問、副詞1問。80%は難易度が非常に高い語彙を使用。

- **なぞ**
- **きせい**
- **かわかす**
- **うけとる**
- **じゅうだい**
- **じつに**

### word_meaning

11問：名詞4問、動詞4問、形容詞2問、副詞1問。80%は難易度が非常に高い語彙を使用。

- **ふじん**
- **ばめん**
- **しんぱん**
- **きょうぎ**
- **かえる**
- **もとめる**
- **つなぐ**
- **あきらめる**
- **きみょう**
- **じゅうだい**
- **たまたま**

### synonym_substitution

5問：名詞2問、動詞2問、形容詞1問。80%は難易度が非常に

## N3 Outline Preview

In [11]:
display(Markdown(n3_outline.as_str))

# 日本語能力試験 JLPT N3 模擬試験

## 第1部：語彙

### kanji_reading

8問：名詞3問、動詞3問、形容詞1問、副詞1問。80%は難易度が非常に高い語彙を使用。すべて漢字表記。

- **ぶんせき**
- **してん**
- **ろんぶん**
- **つかむ**
- **まじる**
- **あきらめる**
- **しんせん**
- **たまたま**

### write_kanji

6問：名詞2問、動詞2問、形容詞1問、副詞1問。80%は難易度が非常に高い語彙を使用。

- **なぞ**
- **きせい**
- **かわかす**
- **うけとる**
- **じゅうだい**
- **じつに**

### word_meaning

11問：名詞4問、動詞4問、形容詞2問、副詞1問。80%は難易度が非常に高い語彙を使用。

- **ふじん**
- **ばめん**
- **しんぱん**
- **きょうぎ**
- **かえる**
- **もとめる**
- **つなぐ**
- **あきらめる**
- **きみょう**
- **じゅうだい**
- **たまたま**

### synonym_substitution

5問：名詞2問、動詞2問、形容詞1問。80%は難易度が非常に高い語彙を使用。

- **かてい**
- **しんり**
- **つかむ**
- **まじる**
- **しんせん**

### word_usage

5問：名詞3問、動詞2問。80%は難易度が非常に高い語彙を使用。すべて漢字表記。

- **ぶんせき**
- **してん**
- **ろんぶん**
- **つかむ**
- **まじる**

## 第2部：文法

### sentence_grammar

13問：敬語1問、副詞1問、助詞1問、その他10問は異なる文型。各文型はTopicListからランダムに選択。

- **レストランで食べ物を注文する**...てほしい
- **週末の予定について話す**...予定だ
- **交通状況について話す**...によって
- **健康とフィットネスについて話す**...ようにする
- **家族について話す**...ことになっている
- **趣味について話す**...つもりだ
- **旅行の計画について話す**...たら
- **最近の映画について話す**...そうだ
- **スポーツについて話す**...みたいだ
- **技術について話す**...によると
- **音楽について話す**...らしい
- **芸術と文化について話す**...ようだ
- **教育について話す**...べきだ

### sentence_sort

5問：文の並び替え問題。TopicListからランダムに選択。

- **家事の分担について話す**...てからでないと
- **友人との日常会話**...たばかり
- **新しいスキルを学ぶ計画について話す**...ようにしている
- **引っ越しの準備について話す**...ついでに
- **日本の祭りや文化イベントについて話す**...たびに

### sentence_structure

1問：文章全体の内容を考えて、文中に4つの異なる文法点を統合した問題。TopicListからランダムに選択。

- **環境問題について話す**...ながら／...ために／...ように／...つもりだ

## 第3部：読解

### short_passage_mail_read

1記事：メール文を読んで質問に答える。TopicListからランダムに選択。

- **領収書を求める**

### short_passage_notification_read

1記事：通知文を読んで質問に答える。TopicListからランダムに選択。

- **バスの時刻表を尋ねる**

### short_passage_narrative_read

2記事：短い物語文を読んで質問に答える。TopicListからランダムに選択。

- **友人へのプレゼント選びについて話す**
- **特別な日の計画について話す**

### midsize_passage_read

2記事：中程度の長さの文章を読んで質問に答える。TopicListからランダムに選択。

- **健康診断や医者への訪問について話す**
- **家事の分担について話す**

### long_passage_read

1記事：長文を読んで質問に答える。TopicListからランダムに選択。

- **異文化交流について話す**

### info_retrieval

1記事：情報検索型の読解問題。TopicListからランダムに選択。

- **公共施設の利用方法について話す**

## 第4部：聴解

### topic_understanding_img

2問：画像を見て話を聞き、内容理解を問う。TopicListからランダムに選択。

- **店で価格を尋ねる**
- **購入したい商品の説明**

### topic_understanding_txt

4問：文章を聞いて内容理解を問う。TopicListからランダムに選択。

- **割引交渉**
- **レストランで食べ物を注文する**
- **料理を褒める**
- **道を尋ねる**

### keypoint_understanding

6問：話の要点を聞き取る問題。TopicListからランダムに選択。

- **交通手段について話す**
- **電車の切符を買う**
- **通勤について説明する**
- **天気の状況について話す**
- **週末の予定について話す**
- **おすすめを尋ねる**

### summary_understanding

3問：話全体の要約理解を問う問題。TopicListからランダムに選択。

- **ショッピング体験を説明する**
- **支払い方法について話す**
- **お気に入りのレストランについて話す**

### active_expression

4問：会話の中での積極的な表現を問う問題。TopicListからランダムに選択。

- **趣味について話す**
- **仕事のプロジェクトについて話す**
- **家族について話す**
- **旅行の計画について話す**

### immediate_ack

9問：即座の応答を問う問題。TopicListからランダムに選択。

- **最近の映画について話す**
- **本について話す**
- **スポーツについて話す**
- **健康とフィットネスについて話す**
- **技術について話す**
- **時事問題について話す**
- **音楽について話す**
- **芸術と文化について話す**
- **教育について話す**

## N3 Exam Result

In [12]:
html_output = render_to_html(n3_exam_paper['sections'])
display(HTML(html_output))

部屋名,定員,利用可能時間,主な利用目的,料金（1時間）
多目的ホール,100人,9:00～21:00,発表会、講演会、展示会,"2,000円"
会議室A,30人,9:00～17:00,会議、勉強会,800円
会議室B,15人,13:00～21:00,打ち合わせ、少人数の集まり,500円
和室,20人,9:00～17:00,茶道、書道、伝統文化活動,600円


In [13]:
timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
filename = f"./output/JLPT_N3_{timestamp}.html"

with open(filename, "w", encoding="utf-8") as file:
    file.write(html_output)

# N4 Level Exam

In [14]:
# runner = TaskRunner(level="N4", exam_type="fast_exam")
# n4_outline, n4_exam_paper = runner.run()

## N4 Outline Preview

In [15]:
# display(Markdown(n4_outline.as_str))

## N4 Exam Result

In [16]:
# html_output = render_to_html(n4_exam_paper['sections'])
# display(HTML(html_output))

# N5 Level Exam

In [17]:
# runner = TaskRunner(level="N5", exam_type="fast_exam")
# n5_outline, n5_exam_paper = runner.run()

## N5 Outline Preview

In [18]:
# display(Markdown(n5_outline.as_str))

## N5  Exam Result

In [19]:
# html_output = render_to_html(n5_exam_paper['sections'])
# display(HTML(html_output))